# 5 · plot — training-data composition

Draws [`5_make_training_composition_data.ipynb`](5_make_training_composition_data.ipynb)'s table
as one bar per training corpus, at the shared figure-1 panel size.

Bar length is tokens, which under the `m2` mixture is also the sampling weight; the structure
count rides on the axis label because it is the other unit the corpus is naturally measured in.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
DATASET = "5_training_composition"
DPI = 300
metadata = figlib.describe(DATASET)

In [ ]:
# --- the training corpus, by source ---------------------------------------------------------------
# Figure 1b. Same panel size and the same explicit-rectangle placement as the map panels, so the
# assembled 2x2 has four identical cells.
import matplotlib.pyplot as plt
import pandas as pd

figlib.figure_style(DPI)
directory = figlib.require(DATASET, "sources.csv")
sources = pd.read_csv(directory / "sources.csv")
budget = figlib.load_metadata(DATASET)["budget"]

BAR_COLOR = "#7A8DA6"      # figure 2's neutral: this panel is a composition, not a contest
PLOT_RECT = (0.30, 0.30, 0.63, 0.52)   # room on the left for two-line source labels
BILLION = 1e9

frame = sources.sort_values("tokens")   # barh draws the first row at the bottom
labels = [f"{row.label}\n{row.documents / 1e6:.1f}M structures" for row in frame.itertuples()]

figure = plt.figure(figsize=figlib.FIG1_PANEL)
axis = figure.add_axes(PLOT_RECT)
axis.barh(labels, frame.tokens / BILLION, height=0.5, color=BAR_COLOR)
for y, row in enumerate(frame.itertuples()):
    axis.text(row.tokens / BILLION + 1.5, y, f"{row.tokens / BILLION:.1f}B", va="center",
              fontsize=7.5, color="0.25")
axis.set(xlabel="training tokens (billions)",
         xlim=(0, 1.20 * frame.tokens.max() / BILLION))
axis.grid(axis="x", alpha=0.25, lw=0.6)
axis.set_axisbelow(True)
axis.tick_params(axis="y", length=0)

# The context the bars cannot carry: how much of the corpus the run actually saw, and that these
# are the decontaminated corpora rather than the originals.
figure.text(0.005, 0.005,
            f"{budget['corpus_tokens'] / BILLION:.1f}B tokens · "
            f"{budget['passes']:.2f} passes · decontaminated",
            ha="left", va="bottom", fontsize=7.5, color="0.35")

figlib.save_figure(figure, "training_composition", DPI, tight=False)
plt.show()
print(frame[["label", "documents_before", "documents_dropped", "documents", "tokens",
             "token_share"]].to_string(index=False))